# Does forward selection find a smaller set that performs as well?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [16]:
import json
import time

import lightgbm as lgb
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

In [17]:
def get_raw_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
v_cols = [c for c in df.columns if c.startswith("V")]

# Rebuild the subgroups and representatives
def get_references_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "references").exists():
            return current / "references"
        current = current.parent
    return Path("../../references")

references_dir = get_references_dir()
with open(references_dir / "column-groups-v.json", "r") as f:
    col_groups_json = json.load(f)

subgroups = []
assigned_cols = set()
for block in col_groups_json['blocks']:
    for group in block['groups']:
        subgroups.append(group)
        assigned_cols.update(group)

unassigned = set(v_cols) - assigned_cols
for u in unassigned:
    subgroups.append([u])

representatives = []
for group in subgroups:
    if len(group) == 1:
        representatives.append(group[0])
    else:
        uniques = [(c, df[c].n_unique()) for c in group]
        best_col = max(uniques, key=lambda x: x[1])[0]
        representatives.append(best_col)


In [18]:
df_pd = df[v_cols].to_pandas()
df_imputed = df_pd.fillna(df_pd.median())

start_time = time.time()
explained_variances = []
pca_exceptions = []
multi_groups = 0
holds_80 = 0

for group in subgroups:
    if len(group) == 1:
        continue
    
    multi_groups += 1
    pca = PCA(n_components=1)
    pca.fit(df_imputed[group])
    var = pca.explained_variance_ratio_[0]
    explained_variances.append(var)
    
    if var > 0.8:
        holds_80 += 1
    else:
        pca_exceptions.append({
            "Group": "+".join(group) if len(group) <= 4 else f"{group[0]}+{group[1]}+{group[2]}…+{group[-1]}",
            "Columns": len(group),
            "Explained variance": var
        })

runtime = time.time() - start_time


In [19]:
fig = px.histogram(x=explained_variances, nbins=20, 
                   title='PCA Explained Variance Ratio across V-Column Groups',
                   labels={'x': 'Explained Variance by First Component', 'y': 'Number of Groups'})
fig.update_yaxes(title_text='Number of Groups')
fig.add_vline(x=0.8, line_dash='dash', line_color='red', annotation_text='80% variance')
fig.update_layout(showlegend=False, template='plotly_white')
fig.show()


In [20]:
summary = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Columns in | {len(v_cols)} |\n"
    f"| Components out | **{len(subgroups)}** |\n"
    f"| Runtime | {runtime:.1f} s |\n"
    f"| Explained variance, median | **{np.median(explained_variances):.3f}** |\n"
    f"| … minimum | {min(explained_variances):.3f} |\n"
    f"| Multi-column groups whose first component holds > 80% | **{holds_80} of {multi_groups} ({holds_80/multi_groups*100:.0f}%)** |\n\n"
    f"### Exceptions\n\n"
)

df_ex = pl.DataFrame(pca_exceptions).sort("Explained variance")
summary += df_ex.to_pandas().to_markdown(index=False)
display(Markdown(summary))

| Metric | Value |
| --- | ---: |
| Columns in | 339 |
| Components out | **129** |
| Runtime | 0.6 s |
| Explained variance, median | **0.946** |
| … minimum | 0.734 |
| Multi-column groups whose first component holds > 80% | **92 of 93 (99%)** |

### Exceptions

| Group               |   Columns |   Explained variance |
|:--------------------|----------:|---------------------:|
| V108+V109+V110+V114 |         4 |             0.733727 |

### Forward Selection

Greedily select features that improve the holdout score the most, stopping when the marginal gain falls below a threshold.

Executed on 129 representative columns over a split of 60,000 rows (train) and 60,000 rows (validation):

In [21]:
target = df["isFraud"].to_numpy()

X_train = df_pd.iloc[:60000][representatives]
y_train = target[:60000]
X_valid = df_pd.iloc[60000:120000][representatives]
y_valid = target[60000:120000]

def eval_feature_set(features):
    if not features:
        return 0.5
    model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, n_jobs=8, random_state=42, verbose=-1)
    model.fit(X_train[features], y_train)
    preds = model.predict_proba(X_valid[features])[:, 1]
    return roc_auc_score(y_valid, preds)

selected = []
remaining = set(representatives)
current_score = 0.5
min_gain = 0.001

history = []
start_time = time.time()

for step in range(1, len(representatives) + 1):
    best_gain = 0
    best_col = None
    best_score = current_score
    
    for col in remaining:
        score = eval_feature_set(selected + [col])
        gain = score - current_score
        if gain > best_gain:
            best_gain = gain
            best_col = col
            best_score = score
            
    if best_gain < min_gain:
        stopped_because = f"gain +{best_gain:.5f} fell below `min_gain` {min_gain}"
        break
        
    selected.append(best_col)
    remaining.remove(best_col)
    current_score = best_score
    history.append({"Step": step, "Column": best_col, "AUC": current_score, "Gain": best_gain})

runtime = time.time() - start_time


#### Forward Selection Trajectory

In [23]:
steps = [h["Step"] for h in history]
aucs = [h["AUC"] for h in history]
fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=aucs, mode='lines+markers', name='ROC-AUC'))
fig.update_layout(title='Forward Selection: Holdout ROC-AUC vs Features Added',
                  xaxis_title='Number of Features (Step)',
                  yaxis_title='Holdout ROC-AUC',
                  template='plotly_white')
fig.show()


#### Forward Selection Results Summary

In [25]:
fs_summary = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Candidates | {len(representatives)} |\n"
    f"| Selected | **{len(selected)}** |\n"
    f"| Holdout ROC-AUC | **{current_score:.4f}** |\n"
    f"| Runtime | {runtime:.0f} s |\n"
    f"| Stopped because | {stopped_because} |\n\n"
    f"### History\n\n"
)

fs_summary += pl.DataFrame(history).to_pandas().to_markdown(index=False)
display(Markdown(fs_summary))

| Metric | Value |
| --- | ---: |
| Candidates | 129 |
| Selected | **11** |
| Holdout ROC-AUC | **0.8460** |
| Runtime | 229 s |
| Stopped because | gain +0.00084 fell below `min_gain` 0.001 |

### History

|   Step | Column   |      AUC |       Gain |
|-------:|:---------|---------:|-----------:|
|      1 | V30      | 0.68078  | 0.18078    |
|      2 | V187     | 0.748509 | 0.0677281  |
|      3 | V294     | 0.794626 | 0.0461178  |
|      4 | V156     | 0.820004 | 0.0253781  |
|      5 | V44      | 0.828804 | 0.00879948 |
|      6 | V283     | 0.833176 | 0.0043716  |
|      7 | V285     | 0.837232 | 0.00405657 |
|      8 | V62      | 0.8404   | 0.00316783 |
|      9 | V198     | 0.842105 | 0.00170521 |
|     10 | V335     | 0.844272 | 0.00216727 |
|     11 | V86      | 0.846034 | 0.00176149 |

The search stopped strictly because the marginal gain fell below the floor. The use of a minimum gain floor acts as an implicit regularizer, preventing the selection of features that purely fit to noise.

**Note:** A ROC-AUC of ~0.8460 generated solely by V-columns over this specific holdout confirms that the block contains strong signal. However, it does not suggest relying exclusively on this narrow subset of features.